In [1]:
from pathlib import Path
import os
import json
import boto3

PROJECT_ROOT = Path(
    "/home/sagemaker-user/AAI-540-Project/"
    "multilingual-it-ticket-routing-mlops"
)
os.chdir(PROJECT_ROOT)

config_path = PROJECT_ROOT / "project_config.json"
project_config = json.loads(config_path.read_text(encoding="utf-8"))

AWS_REGION = project_config["aws_region"]
PROJECT_BUCKET = project_config["s3_bucket"]

aws_session = boto3.Session(region_name=AWS_REGION)
sagemaker_client = aws_session.client("sagemaker")
s3_client = aws_session.client("s3")

PIPELINE_SOURCE_DIR = PROJECT_ROOT / "src" / "ticket_pipeline"
PIPELINE_REPORT_DIR = PROJECT_ROOT / "reports" / "pipeline"
PIPELINE_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Inspect the successful training job so we can reuse its configuration.
training_details = sagemaker_client.describe_training_job(
    TrainingJobName=project_config["final_model"]["training_job_name"]
)
assert training_details["TrainingJobStatus"] == "Completed"

pipeline_role = training_details["RoleArn"]
pipeline_image = training_details["AlgorithmSpecification"]["TrainingImage"]
feature_datasets = project_config["feature_store"]["datasets"]
model_group_name = project_config["model_registry"]["model_package_group_name"]

print("Pipeline source folder:", PIPELINE_SOURCE_DIR)
print("Execution role:", pipeline_role)
print("Training image:", pipeline_image)
print("Registry group:", model_group_name)

for split in ["train", "validation"]:
    print(f"{split} rows:", feature_datasets[split]["rows"])

print("\nExisting training script configuration:")
for key in ["sagemaker_program", "sagemaker_submit_directory"]:
    print(key + ":", training_details.get("HyperParameters", {}).get(key))

print("\nPipeline setup ready. No jobs started.")

Pipeline source folder: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/src/ticket_pipeline
Execution role: arn:aws:iam::390568313988:role/LabRole
Training image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3
Registry group: aai540-group3-ticket-routing
train rows: 17710
validation rows: 4428

Existing training script configuration:
sagemaker_program: "train.py"
sagemaker_submit_directory: "s3://aai540-group3-tickets-390568313988-us-east-1/training-code/aai540-group3-lr-20261004-003353/source/sourcedir.tar.gz"

Pipeline setup ready. No jobs started.


In [2]:
import hashlib
import tarfile
from io import BytesIO
from urllib.parse import urlparse

# Retrieve the source package used by the successful training job.
training_source_uri = json.loads(
    training_details["HyperParameters"]["sagemaker_submit_directory"]
)
source_location = urlparse(training_source_uri)

response = s3_client.get_object(
    Bucket=source_location.netloc,
    Key=source_location.path.lstrip("/"),
)

source_version = response.get("VersionId")
assert source_version and source_version != "null"

with response["Body"] as body:
    training_source_bytes = body.read()

expected_hashes = project_config["managed_training"]["source_sha256"]
verified_source_files = {}

with tarfile.open(
    fileobj=BytesIO(training_source_bytes), mode="r:gz"
) as archive:
    for filename, expected_hash in expected_hashes.items():
        matches = [
            member for member in archive.getmembers()
            if member.isfile() and Path(member.name).name == filename
        ]
        assert len(matches) == 1, f"Missing or ambiguous file: {filename}"

        extracted = archive.extractfile(matches[0])
        assert extracted is not None
        with extracted:
            content = extracted.read()

        assert hashlib.sha256(content).hexdigest() == expected_hash, (
            f"Training source changed: {filename}"
        )
        verified_source_files[filename] = content
        print("Verified training source:", filename)

frozen_training_config = json.loads(
    verified_source_files["training_config.json"]
)
assert frozen_training_config == project_config["model_development"]

training_source_artifact = {
    "uri": training_source_uri,
    "version_id": source_version,
    "sha256": hashlib.sha256(training_source_bytes).hexdigest(),
    "file_sha256": expected_hashes,
}

pipeline_config = {
    "aws_region": AWS_REGION,
    "datasets": {
        split: feature_datasets[split]
        for split in ["train", "validation"]
    },
    "training_configuration": frozen_training_config,
    "training_source": training_source_artifact,
    "training_image": pipeline_image,
    "execution_role": pipeline_role,
    "model_package_group_name": model_group_name,
    "serving_assets": project_config["model_registry"]["verified_assets"],
    "quality_gate": {
        "minimum_validation_macro_f1": 0.40,
        "require_accuracy_above_benchmark": True,
        "registration_approval_status": "PendingManualApproval",
        "purpose": "Course demonstration gate; not operational approval.",
    },
}

pipeline_config_path = PIPELINE_SOURCE_DIR / "pipeline_config.json"
pipeline_config_path.write_text(
    json.dumps(pipeline_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

(PIPELINE_REPORT_DIR / "training_source_verification.json").write_text(
    json.dumps(training_source_artifact, indent=2),
    encoding="utf-8",
)

print("\nPipeline inputs: training and validation only.")
print("Selected model settings preserved.")
print("Pipeline configuration saved:", pipeline_config_path)
print("No jobs started.")

Verified training source: train.py
Verified training source: requirements.txt
Verified training source: training_config.json

Pipeline inputs: training and validation only.
Selected model settings preserved.
Pipeline configuration saved: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/src/ticket_pipeline/pipeline_config.json
No jobs started.


In [3]:
%%writefile src/ticket_pipeline/verify_data.py
import argparse
import csv
import hashlib
import io
import json
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import boto3


def require(condition, message):
    if not condition:
        raise ValueError(message)


def verify_dataset(s3, split_name, reference):
    location = urlparse(reference["uri"])
    require(
        location.scheme == "s3" and bool(location.netloc),
        f"{split_name}: invalid S3 URI."
    )
    require(
        reference.get("version_id") not in (None, "", "null"),
        f"{split_name}: a saved S3 version is required."
    )

    response = s3.get_object(
        Bucket=location.netloc,
        Key=location.path.lstrip("/"),
        VersionId=reference["version_id"],
    )
    try:
        content = response["Body"].read()
    finally:
        response["Body"].close()

    require(
        response.get("VersionId") == reference["version_id"],
        f"{split_name}: S3 version mismatch."
    )

    fingerprint = hashlib.sha256(content).hexdigest()
    require(
        fingerprint == reference["sha256"],
        f"{split_name}: dataset fingerprint mismatch."
    )

    reader = csv.DictReader(
        io.StringIO(content.decode("utf-8-sig"), newline="")
    )
    required_columns = {"ticket_id", "combined_text", "queue"}
    require(
        required_columns.issubset(set(reader.fieldnames or [])),
        f"{split_name}: required columns are missing."
    )
    rows = list(reader)

    require(
        len(rows) == int(reference["rows"]) and len(rows) > 0,
        f"{split_name}: unexpected row count."
    )

    ticket_ids = []
    queues = set()

    for row_number, row in enumerate(rows, start=2):
        ticket_id = row.get("ticket_id")
        text = row.get("combined_text")
        queue = row.get("queue")

        require(
            isinstance(ticket_id, str) and bool(ticket_id.strip()),
            f"{split_name}: missing ticket ID at row {row_number}."
        )
        require(
            isinstance(text, str) and bool(text.strip()),
            f"{split_name}: empty text at row {row_number}."
        )
        require(
            isinstance(queue, str) and bool(queue.strip()),
            f"{split_name}: missing queue at row {row_number}."
        )

        ticket_ids.append(ticket_id)
        queues.add(queue)

    require(
        len(set(ticket_ids)) == len(ticket_ids),
        f"{split_name}: duplicate ticket IDs."
    )

    summary = {
        "uri": reference["uri"],
        "version_id": reference["version_id"],
        "sha256": fingerprint,
        "rows": len(rows),
        "queue_labels": sorted(queues),
    }

    return content, set(ticket_ids), queues, summary


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument(
        "--output-dir",
        default="/opt/ml/processing/output",
    )
    args = parser.parse_args()

    config = json.loads(Path(args.config).read_text())
    datasets = config["datasets"]

    require(
        set(datasets) == {"train", "validation"},
        "This step accepts only training and validation datasets."
    )

    frozen_datasets = config["training_configuration"]["feature_datasets"]
    for split_name in ("train", "validation"):
        require(
            datasets[split_name] == frozen_datasets[split_name],
            f"{split_name}: dataset differs from the training configuration."
        )

    s3 = boto3.client("s3", region_name=config["aws_region"])
    verified = {
        name: verify_dataset(s3, name, datasets[name])
        for name in ("train", "validation")
    }

    require(
        verified["train"][1].isdisjoint(verified["validation"][1]),
        "Training and validation contain overlapping ticket IDs."
    )
    require(
        len(verified["train"][2]) == 10,
        "Training data must contain the project's 10 queues."
    )
    require(
        verified["validation"][2] == verified["train"][2],
        "Training and validation must contain the same 10 queues."
    )

    # Write only after both datasets pass every check.
    # Preserve the original bytes so training can verify the same hashes.
    output_dir = Path(args.output_dir)
    for split_name, result in verified.items():
        destination = output_dir / split_name / f"{split_name}.csv"
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(result[0])

    report = {
        "checked_at": datetime.now(timezone.utc).isoformat(),
        "status": "PASS",
        "ticket_ids_disjoint": True,
        "datasets": {
            name: result[3] for name, result in verified.items()
        },
    }

    report_path = output_dir / "verification" / "data_verification.json"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(json.dumps(report, indent=2) + "\n")

    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

Writing src/ticket_pipeline/verify_data.py


In [4]:
import hashlib
import json
import subprocess
import sys

verification_script = PIPELINE_SOURCE_DIR / "verify_data.py"
pipeline_config_path = PIPELINE_SOURCE_DIR / "pipeline_config.json"
verification_output = PROJECT_ROOT / "data" / "pipeline_preflight"

# Check Python syntax before running the script.
compile(
    verification_script.read_text(),
    str(verification_script),
    "exec",
)
print("Data verification script syntax: valid")

# Run using this notebook's Python environment.
result = subprocess.run(
    [
        sys.executable,
        str(verification_script),
        "--config", str(pipeline_config_path),
        "--output-dir", str(verification_output),
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=180,
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Data verification failed. Share the output above.")

verification_report = json.loads(
    (
        verification_output
        / "verification"
        / "data_verification.json"
    ).read_text()
)

assert verification_report["status"] == "PASS"
assert verification_report["ticket_ids_disjoint"] is True

# Confirm the output files preserve the original dataset fingerprints.
saved_pipeline_config = json.loads(pipeline_config_path.read_text())

for split_name in ("train", "validation"):
    output_file = verification_output / split_name / f"{split_name}.csv"
    actual_hash = hashlib.sha256(output_file.read_bytes()).hexdigest()
    expected_hash = saved_pipeline_config["datasets"][split_name]["sha256"]

    assert actual_hash == expected_hash, f"{split_name}: output mismatch."

    rows = verification_report["datasets"][split_name]["rows"]
    print(f"PASS: {split_name} — {rows:,} tickets; fingerprint preserved.")

PIPELINE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
(PIPELINE_REPORT_DIR / "data_verification.json").write_text(
    json.dumps(verification_report, indent=2) + "\n"
)

print("PASS: training and validation ticket IDs do not overlap.")
print("PASS: both datasets contain the same 10 queues.")
print("Verification report saved. No AWS jobs started.")

Data verification script syntax: valid
PASS: train — 17,710 tickets; fingerprint preserved.
PASS: validation — 4,428 tickets; fingerprint preserved.
PASS: training and validation ticket IDs do not overlap.
PASS: both datasets contain the same 10 queues.
Verification report saved. No AWS jobs started.


In [5]:
%%writefile src/ticket_pipeline/prepare_evaluation.py
import argparse
import hashlib
import io
import json
import math
import tarfile
from pathlib import Path
from urllib.parse import urlparse

import boto3


def require(condition, message):
    if not condition:
        raise ValueError(message)


def download_artifact(s3, uri):
    location = urlparse(uri)
    require(
        location.scheme == "s3" and bool(location.netloc),
        "Invalid S3 artifact URI."
    )

    response = s3.get_object(
        Bucket=location.netloc,
        Key=location.path.lstrip("/"),
    )
    try:
        content = response["Body"].read()
    finally:
        response["Body"].close()

    version_id = response.get("VersionId")
    require(
        version_id not in (None, "", "null"),
        "Artifact must have an S3 version ID."
    )

    reference = {
        "uri": uri,
        "version_id": version_id,
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    return content, reference


def read_archive_json(content, filename):
    # Read the requested JSON directly without extracting the archive.
    with tarfile.open(fileobj=io.BytesIO(content), mode="r:gz") as archive:
        matches = [
            member for member in archive.getmembers()
            if member.isfile() and Path(member.name).name == filename
        ]
        require(
            len(matches) == 1,
            f"Expected exactly one {filename} in the archive."
        )
        with archive.extractfile(matches[0]) as stream:
            return json.load(stream)


def validate_metrics(metrics, description):
    for name in ("accuracy", "macro_f1", "weighted_f1"):
        value = metrics.get(name)
        require(
            type(value) in (int, float)
            and math.isfinite(value)
            and 0 <= value <= 1,
            f"{description}: invalid {name}."
        )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--training-job-name", required=True)
    parser.add_argument(
        "--output-dir",
        default="/opt/ml/processing/output",
    )
    args = parser.parse_args()

    config = json.loads(Path(args.config).read_text())
    session = boto3.Session(region_name=config["aws_region"])
    sagemaker = session.client("sagemaker")
    s3 = session.client("s3")

    job = sagemaker.describe_training_job(
        TrainingJobName=args.training_job_name
    )
    require(
        job["TrainingJobStatus"] == "Completed",
        "The training job has not completed successfully."
    )
    require(
        job["AlgorithmSpecification"]["TrainingImage"]
        == config["training_image"],
        "Training image differs from the pipeline configuration."
    )

    model_uri = job["ModelArtifacts"]["S3ModelArtifacts"]
    output_uri = model_uri.rsplit("/", 1)[0] + "/output.tar.gz"

    model_bytes, model_reference = download_artifact(s3, model_uri)
    output_bytes, output_reference = download_artifact(s3, output_uri)

    metadata = read_archive_json(model_bytes, "model_metadata.json")
    evaluation = read_archive_json(output_bytes, "evaluation.json")

    require(
        metadata["configuration"] == config["training_configuration"],
        "The model settings differ from the frozen training configuration."
    )
    require(
        len(set(metadata["queue_labels"])) == 10,
        "The model must contain 10 distinct queue labels."
    )
    require(
        evaluation["evaluation_split"] == "validation",
        "The pipeline quality gate must use validation results."
    )
    require(
        evaluation["training_rows"] == config["datasets"]["train"]["rows"],
        "Training row count mismatch."
    )
    require(
        evaluation["validation_rows"]
        == config["datasets"]["validation"]["rows"],
        "Validation row count mismatch."
    )

    validate_metrics(evaluation["model"], "Model results")
    validate_metrics(evaluation["benchmark"], "Benchmark results")

    report = {
        "evaluation_split": "validation",
        "training_job_name": args.training_job_name,
        "training_rows": evaluation["training_rows"],
        "evaluated_tickets": evaluation["validation_rows"],
        "multiclass_classification_metrics": {
            name: {"value": evaluation["model"][name]}
            for name in ("accuracy", "macro_f1", "weighted_f1")
        },
        "benchmark": evaluation["benchmark"],
        "accuracy_improvement": (
            evaluation["model"]["accuracy"]
            - evaluation["benchmark"]["accuracy"]
        ),
        "evaluation_dataset": config["datasets"]["validation"],
    }

    assets = {
        "training_job_name": args.training_job_name,
        "model_artifact": model_reference,
        "training_output_artifact": output_reference,
        "training_configuration_verified": True,
        "queue_labels": metadata["queue_labels"],
    }

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for filename, value in (
        ("evaluation.json", report),
        ("model_assets.json", assets),
    ):
        (output_dir / filename).write_text(
            json.dumps(value, indent=2, allow_nan=False) + "\n"
        )

    print(json.dumps(report, indent=2, allow_nan=False))


if __name__ == "__main__":
    main()

Writing src/ticket_pipeline/prepare_evaluation.py


In [6]:
import json
import math
import subprocess
import sys

evaluation_script = PIPELINE_SOURCE_DIR / "prepare_evaluation.py"
evaluation_output = PROJECT_ROOT / "data" / "pipeline_evaluation_preflight"
existing_job_name = project_config["final_model"]["training_job_name"]

compile(
    evaluation_script.read_text(),
    str(evaluation_script),
    "exec",
)
print("Evaluation script syntax: valid")

result = subprocess.run(
    [
        sys.executable,
        str(evaluation_script),
        "--config", str(PIPELINE_SOURCE_DIR / "pipeline_config.json"),
        "--training-job-name", existing_job_name,
        "--output-dir", str(evaluation_output),
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=180,
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Evaluation verification failed. Share the output above.")

evaluation_report = json.loads(
    (evaluation_output / "evaluation.json").read_text()
)
evaluation_assets = json.loads(
    (evaluation_output / "model_assets.json").read_text()
)
saved_results = json.loads(
    (
        PROJECT_ROOT
        / "reports"
        / "managed_training"
        / "evaluation.json"
    ).read_text()
)

assert evaluation_report["evaluation_split"] == "validation"
assert evaluation_report["training_job_name"] == existing_job_name
assert evaluation_report["evaluated_tickets"] == 4428

for metric in ("accuracy", "macro_f1", "weighted_f1"):
    actual = evaluation_report["multiclass_classification_metrics"][metric]["value"]
    expected = saved_results["model"][metric]

    assert math.isclose(actual, expected, rel_tol=0, abs_tol=1e-12), (
        f"{metric}: result differs from the saved validation report."
    )
    print(f"PASS: validation {metric} matches the saved result ({actual:.4f}).")

final_model = project_config["final_model"]
model_reference = evaluation_assets["model_artifact"]

assert model_reference["uri"] == final_model["model_uri"]
assert model_reference["version_id"] == final_model["model_version_id"]
assert model_reference["sha256"] == final_model["model_archive_sha256"]
print("PASS: model artifact version and fingerprint match.")

# Save this as a local verification of the existing training job.
PIPELINE_REPORT_DIR.mkdir(parents=True, exist_ok=True)

for filename, report in (
    ("preflight_evaluation.json", evaluation_report),
    ("preflight_model_assets.json", evaluation_assets),
):
    (PIPELINE_REPORT_DIR / filename).write_text(
        json.dumps(report, indent=2) + "\n"
    )

print("Evaluation verification reports saved.")
print("No retraining, registration, or new AWS jobs started.")

Evaluation script syntax: valid
PASS: validation accuracy matches the saved result (0.4557).
PASS: validation macro_f1 matches the saved result (0.4362).
PASS: validation weighted_f1 matches the saved result (0.4569).
PASS: model artifact version and fingerprint match.
Evaluation verification reports saved.
No retraining, registration, or new AWS jobs started.


In [7]:
import hashlib
import json
from urllib.parse import urlparse

pipeline_config = json.loads(
    (PIPELINE_SOURCE_DIR / "pipeline_config.json").read_text()
)

# Collect the two verified scripts and their configuration.
pipeline_files = {
    filename: (PIPELINE_SOURCE_DIR / filename).read_bytes()
    for filename in (
        "verify_data.py",
        "prepare_evaluation.py",
        "pipeline_config.json",
    )
}

# Retrieve the exact training-code version we verified earlier.
training_source_reference = pipeline_config["training_source"]
source_location = urlparse(training_source_reference["uri"])

response = s3_client.get_object(
    Bucket=source_location.netloc,
    Key=source_location.path.lstrip("/"),
    VersionId=training_source_reference["version_id"],
)
try:
    training_archive = response["Body"].read()
finally:
    response["Body"].close()

assert response.get("VersionId") == training_source_reference["version_id"]
assert (
    hashlib.sha256(training_archive).hexdigest()
    == training_source_reference["sha256"]
), "Training source fingerprint mismatch."

# Use file contents to identify this version of the pipeline code.
file_hashes = {
    name: hashlib.sha256(content).hexdigest()
    for name, content in pipeline_files.items()
}
file_hashes["training/sourcedir.tar.gz"] = (
    training_source_reference["sha256"]
)

pipeline_code_id = hashlib.sha256(
    json.dumps(file_hashes, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

pipeline_code_prefix = f"pipeline-code/{pipeline_code_id}"


def upload_pipeline_file(relative_path, content, content_type):
    key = f"{pipeline_code_prefix}/{relative_path}"

    uploaded = s3_client.put_object(
        Bucket=PROJECT_BUCKET,
        Key=key,
        Body=content,
        ContentType=content_type,
    )

    version_id = uploaded.get("VersionId")
    assert version_id not in (None, "", "null"), (
        f"Missing S3 version ID for {relative_path}."
    )

    return {
        "uri": f"s3://{PROJECT_BUCKET}/{key}",
        "version_id": version_id,
        "sha256": hashlib.sha256(content).hexdigest(),
    }


pipeline_source_artifacts = {}

for filename, content in pipeline_files.items():
    content_type = (
        "application/json" if filename.endswith(".json") else "text/plain"
    )
    pipeline_source_artifacts[filename] = upload_pipeline_file(
        f"scripts/{filename}",
        content,
        content_type,
    )
    print(f"Uploaded: {filename}")

pipeline_training_source = upload_pipeline_file(
    "training/sourcedir.tar.gz",
    training_archive,
    "application/gzip",
)

pipeline_staging = {
    "pipeline_name": "aai540-group3-ticket-routing-pipeline",
    "code_id": pipeline_code_id,
    "code_prefix_uri": (
        f"s3://{PROJECT_BUCKET}/{pipeline_code_prefix}/scripts/"
    ),
    "source_artifacts": pipeline_source_artifacts,
    "training_source": pipeline_training_source,
}

PIPELINE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
(PIPELINE_REPORT_DIR / "staged_sources.json").write_text(
    json.dumps(pipeline_staging, indent=2) + "\n"
)

project_config["ml_pipeline"] = pipeline_staging
(PROJECT_ROOT / "project_config.json").write_text(
    json.dumps(project_config, indent=2) + "\n"
)

print("Verified training code copied to the pipeline's S3 folder.")
print("Pipeline source locations and version IDs saved.")
print("No AWS jobs started.")

Uploaded: verify_data.py
Uploaded: prepare_evaluation.py
Uploaded: pipeline_config.json
Verified training code copied to the pipeline's S3 folder.
Pipeline source locations and version IDs saved.
No AWS jobs started.


In [12]:
import copy
import json

pipeline_staging = json.loads(
    (PIPELINE_REPORT_DIR / "staged_sources.json").read_text()
)
pipeline_config = json.loads(
    (PIPELINE_SOURCE_DIR / "pipeline_config.json").read_text()
)

pipeline_name = pipeline_staging["pipeline_name"]
pipeline_role = pipeline_config["execution_role"]
pipeline_image = pipeline_config["training_image"]


def pipeline_output_path(*parts):
    # Every execution receives its own S3 output folder.
    return {
        "Std:Join": {
            "On": "/",
            "Values": [
                f"s3://{PROJECT_BUCKET}",
                "pipelines",
                pipeline_name,
                {"Get": "Execution.PipelineExecutionId"},
                *parts,
            ],
        }
    }


def processing_arguments(script_name, extra_arguments, outputs):
    return {
        "RoleArn": pipeline_role,
        "AppSpecification": {
            "ImageUri": pipeline_image,
            "ContainerEntrypoint": [
                "python3",
                f"/opt/ml/processing/code/{script_name}",
            ],
            "ContainerArguments": [
                "--config",
                "/opt/ml/processing/code/pipeline_config.json",
                "--output-dir",
                "/opt/ml/processing/output",
                *extra_arguments,
            ],
        },
        "ProcessingInputs": [
            {
                "InputName": "code",
                "S3Input": {
                    "S3Uri": pipeline_staging["code_prefix_uri"],
                    "LocalPath": "/opt/ml/processing/code",
                    "S3DataType": "S3Prefix",
                    "S3InputMode": "File",
                    "S3DataDistributionType": "FullyReplicated",
                },
            }
        ],
        "ProcessingOutputConfig": {"Outputs": outputs},
        "ProcessingResources": {
            "ClusterConfig": {
                "InstanceCount": 1,
                "InstanceType": "ml.m5.xlarge",
                "VolumeSizeInGB": 20,
            }
        },
        "StoppingCondition": {"MaxRuntimeInSeconds": 900},
    }


# Step 1: Verify both datasets and preserve their original CSV bytes.
verify_outputs = [
    {
        "OutputName": name,
        "S3Output": {
            "S3Uri": pipeline_output_path("verified-data", name),
            "LocalPath": f"/opt/ml/processing/output/{name}",
            "S3UploadMode": "EndOfJob",
        },
    }
    for name in ("train", "validation", "verification")
]

verify_step = {
    "Name": "VerifyData",
    "Type": "Processing",
    "Arguments": processing_arguments(
        "verify_data.py",
        [],
        verify_outputs,
    ),
}

# Step 2: Reuse the successful training job's script settings.
training_details = sagemaker_client.describe_training_job(
    TrainingJobName=project_config["final_model"]["training_job_name"]
)
assert training_details["TrainingJobStatus"] == "Completed"

training_hyperparameters = copy.deepcopy(
    training_details["HyperParameters"]
)
training_hyperparameters["sagemaker_submit_directory"] = json.dumps(
    pipeline_staging["training_source"]["uri"]
)
training_hyperparameters.pop("sagemaker_job_name", None)

training_algorithm = {
    "TrainingImage": pipeline_image,
    "TrainingInputMode": "File",
}
if training_details["AlgorithmSpecification"].get("MetricDefinitions"):
    training_algorithm["MetricDefinitions"] = copy.deepcopy(
        training_details["AlgorithmSpecification"]["MetricDefinitions"]
    )

training_channels = [
    {
        "ChannelName": name,
        "ContentType": "text/csv",
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": {
                    "Get": (
                        f'Steps.VerifyData.ProcessingOutputConfig.'
                        f"Outputs['{name}'].S3Output.S3Uri"
                    )
                },
                "S3DataDistributionType": "FullyReplicated",
            }
        },
    }
    for name in ("train", "validation")
]

train_step = {
    "Name": "TrainModel",
    "Type": "Training",
    "DependsOn": ["VerifyData"],
    "Arguments": {
        "RoleArn": pipeline_role,
        "AlgorithmSpecification": training_algorithm,
        "HyperParameters": training_hyperparameters,
        "InputDataConfig": training_channels,
        "OutputDataConfig": {
            "S3OutputPath": pipeline_output_path("training")
        },
        "ResourceConfig": {
            "InstanceCount": 1,
            "InstanceType": "ml.m5.xlarge",
            "VolumeSizeInGB": 20,
        },
        "StoppingCondition": {"MaxRuntimeInSeconds": 900},
        "ProfilerConfig": {"DisableProfiler": True},
    },
}

# Step 3: Read validation results from this pipeline's training job.
evaluation_outputs = [
    {
        "OutputName": "evaluation",
        "S3Output": {
            "S3Uri": pipeline_output_path("evaluation"),
            "LocalPath": "/opt/ml/processing/output",
            "S3UploadMode": "EndOfJob",
        },
    }
]

evaluation_step = {
    "Name": "PrepareEvaluation",
    "Type": "Processing",
    "DependsOn": ["TrainModel"],
    "Arguments": processing_arguments(
        "prepare_evaluation.py",
        [
            "--training-job-name",
            {"Get": "Steps.TrainModel.TrainingJobName"},
        ],
        evaluation_outputs,
    ),
    "PropertyFiles": [
        {
            "PropertyFileName": "ValidationReport",
            "OutputName": "evaluation",
            "FilePath": "evaluation.json",
        }
    ],
}

core_steps = [verify_step, train_step, evaluation_step]

(PIPELINE_REPORT_DIR / "core_steps.json").write_text(
    json.dumps(core_steps, indent=2) + "\n"
)

for step in core_steps:
    print(f"Defined: {step['Name']} ({step['Type']})")

print("Compute per job: one ml.m5.xlarge instance.")
print("Maximum runtime per job: 15 minutes.")
print("Core pipeline steps saved locally. No AWS jobs started.")

Defined: VerifyData (Processing)
Defined: TrainModel (Training)
Defined: PrepareEvaluation (Processing)
Compute per job: one ml.m5.xlarge instance.
Maximum runtime per job: 15 minutes.
Core pipeline steps saved locally. No AWS jobs started.


In [13]:
import copy
import hashlib
import json

core_steps = json.loads(
    (PIPELINE_REPORT_DIR / "core_steps.json").read_text()
)
pipeline_config = json.loads(
    (PIPELINE_SOURCE_DIR / "pipeline_config.json").read_text()
)
pipeline_staging = json.loads(
    (PIPELINE_REPORT_DIR / "staged_sources.json").read_text()
)

serving_assets = pipeline_config["serving_assets"]
quality_gate = pipeline_config["quality_gate"]

assert quality_gate["registration_approval_status"] == "PendingManualApproval"
assert quality_gate["require_accuracy_above_benchmark"] is True


def validation_value(json_path):
    # Read a value from the evaluation step's saved JSON report.
    return {
        "Std:JsonGet": {
            "PropertyFile": {
                "Get": (
                    "Steps.PrepareEvaluation."
                    "PropertyFiles.ValidationReport"
                )
            },
            "Path": json_path,
        }
    }


evaluation_report_uri = {
    "Std:Join": {
        "On": "/",
        "Values": [
            {
                "Get": (
                    "Steps.PrepareEvaluation.ProcessingOutputConfig."
                    "Outputs['evaluation'].S3Output.S3Uri"
                )
            },
            "evaluation.json",
        ],
    }
}

# Register the NEW model produced by this pipeline execution.
register_step = {
    "Name": "RegisterModelForReview",
    "Type": "RegisterModel",
    "Arguments": {
        "ModelPackageGroupName": (
            pipeline_config["model_package_group_name"]
        ),
        "ModelPackageDescription": (
            "Pipeline-trained ticket-routing model. "
            "Passed validation checks; requires team review."
        ),
        "ModelApprovalStatus": "PendingManualApproval",
        "InferenceSpecification": {
            "Containers": [
                {
                    "Image": serving_assets["serving_image"],
                    "ModelDataUrl": {
                        "Get": (
                            "Steps.TrainModel."
                            "ModelArtifacts.S3ModelArtifacts"
                        )
                    },
                    "Environment": copy.deepcopy(
                        serving_assets["serving_environment"]
                    ),
                }
            ],
            "SupportedTransformInstanceTypes": ["ml.m5.xlarge"],
            "SupportedContentTypes": ["application/jsonlines"],
            "SupportedResponseMIMETypes": ["application/jsonlines"],
        },
        "ModelMetrics": {
            "ModelQuality": {
                "Statistics": {
                    "ContentType": "application/json",
                    "S3Uri": evaluation_report_uri,
                }
            }
        },
        "CustomerMetadataProperties": {
            "pipeline_code_id": pipeline_staging["code_id"],
            "evaluation_split": "validation",
            "selection_metric": "validation_macro_f1",
        },
    },
}

fail_step = {
    "Name": "StopForQualityReview",
    "Type": "Fail",
    "Arguments": {
        "ErrorMessage": (
            "Validation quality gate failed. "
            "Macro F1 must meet the configured minimum, "
            "and accuracy must exceed the majority-class benchmark. "
            "No model was registered."
        )
    },
}

gate_step = {
    "Name": "CheckValidationQuality",
    "Type": "Condition",
    "DependsOn": ["PrepareEvaluation"],
    "Arguments": {
        "Conditions": [
            {
                "Type": "GreaterThanOrEqualTo",
                "LeftValue": validation_value(
                    "multiclass_classification_metrics.macro_f1.value"
                ),
                "RightValue": {
                    "Get": "Parameters.MinimumValidationMacroF1"
                },
            },
            {
                "Type": "GreaterThan",
                "LeftValue": validation_value("accuracy_improvement"),
                "RightValue": 0.0,
            },
        ],
        "IfSteps": [register_step],
        "ElseSteps": [fail_step],
    },
}

pipeline_definition = {
    "Version": "2020-12-01",
    "Parameters": [
        {
            "Name": "MinimumValidationMacroF1",
            "Type": "Float",
            "DefaultValue": quality_gate["minimum_validation_macro_f1"],
        }
    ],
    "Steps": core_steps + [gate_step],
}

definition_text = json.dumps(
    pipeline_definition, indent=2, allow_nan=False
) + "\n"

definition_path = PIPELINE_SOURCE_DIR / "pipeline_definition.json"
definition_path.write_text(definition_text)

project_config["ml_pipeline"].update({
    "definition_file": str(definition_path.relative_to(PROJECT_ROOT)),
    "definition_sha256": hashlib.sha256(
        definition_text.encode("utf-8")
    ).hexdigest(),
    "quality_gate": quality_gate,
})

(PROJECT_ROOT / "project_config.json").write_text(
    json.dumps(project_config, indent=2) + "\n"
)

print("Pipeline:", pipeline_staging["pipeline_name"])
print("Main steps:")
for step in pipeline_definition["Steps"]:
    print(f"  - {step['Name']}")

print(
    "Minimum validation macro F1:",
    quality_gate["minimum_validation_macro_f1"],
)
print("Accuracy must exceed the majority-class benchmark.")
print("Passing outcome: register model as PendingManualApproval.")
print("Failing outcome: stop without model registration.")
print("Complete definition saved:", definition_path)
print("Not yet submitted to SageMaker. No jobs started.")

Pipeline: aai540-group3-ticket-routing-pipeline
Main steps:
  - VerifyData
  - TrainModel
  - PrepareEvaluation
  - CheckValidationQuality
Minimum validation macro F1: 0.4
Accuracy must exceed the majority-class benchmark.
Passing outcome: register model as PendingManualApproval.
Failing outcome: stop without model registration.
Complete definition saved: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/src/ticket_pipeline/pipeline_definition.json
Not yet submitted to SageMaker. No jobs started.


In [14]:
import hashlib
import json
import uuid

definition_path = PIPELINE_SOURCE_DIR / "pipeline_definition.json"
definition_text = definition_path.read_text()
local_definition = json.loads(definition_text)

pipeline_name = project_config["ml_pipeline"]["pipeline_name"]
definition_hash = hashlib.sha256(
    definition_text.encode("utf-8")
).hexdigest()

pipeline_request = {
    "PipelineName": pipeline_name,
    "PipelineDisplayName": "AAI540-Group3-Ticket-Routing",
    "PipelineDescription": (
        "Verify versioned datasets, train the selected model, "
        "check validation quality, and register passing models "
        "for manual review."
    ),
    "PipelineDefinition": definition_text,
    "RoleArn": pipeline_config["execution_role"],
    "ParallelismConfiguration": {
        "MaxParallelExecutionSteps": 1
    },
}

# Reuse the existing pipeline if this cell has already succeeded.
try:
    existing_pipeline = sagemaker_client.describe_pipeline(
        PipelineName=pipeline_name
    )
except sagemaker_client.exceptions.ResourceNotFound:
    existing_pipeline = None

if existing_pipeline is None:
    response = sagemaker_client.create_pipeline(
        **pipeline_request,
        ClientRequestToken=str(
            uuid.uuid5(
                uuid.NAMESPACE_URL,
                f"{pipeline_name}:{definition_hash}",
            )
        ),
    )
    action = "Created"

elif (
    json.loads(existing_pipeline["PipelineDefinition"]) == local_definition
    and existing_pipeline["RoleArn"] == pipeline_request["RoleArn"]
    and existing_pipeline.get("ParallelismConfiguration")
    == pipeline_request["ParallelismConfiguration"]
):
    action = "Reused unchanged"

else:
    response = sagemaker_client.update_pipeline(**pipeline_request)
    action = "Updated"

# Read back what SageMaker actually saved.
saved_pipeline = sagemaker_client.describe_pipeline(
    PipelineName=pipeline_name
)

assert (
    json.loads(saved_pipeline["PipelineDefinition"]) == local_definition
), "The saved pipeline definition does not match the local definition."

assert saved_pipeline["RoleArn"] == pipeline_request["RoleArn"]
assert saved_pipeline["PipelineStatus"] == "Active"

registration_report = {
    "pipeline_name": pipeline_name,
    "pipeline_arn": saved_pipeline["PipelineArn"],
    "status": saved_pipeline["PipelineStatus"],
    "definition_sha256": definition_hash,
    "definition_matches": True,
    "execution_role": saved_pipeline["RoleArn"],
    "maximum_parallel_steps": 1,
    "action": action,
}

(PIPELINE_REPORT_DIR / "pipeline_registration.json").write_text(
    json.dumps(registration_report, indent=2) + "\n"
)

project_config["ml_pipeline"].update({
    "pipeline_arn": saved_pipeline["PipelineArn"],
    "status": saved_pipeline["PipelineStatus"],
    "definition_sha256": definition_hash,
    "definition_registered": True,
})

(PROJECT_ROOT / "project_config.json").write_text(
    json.dumps(project_config, indent=2) + "\n"
)

print(f"{action}: {pipeline_name}")
print("Status:", saved_pipeline["PipelineStatus"])
print("PASS: SageMaker's saved definition matches the local definition.")
print("Pipeline ARN:", saved_pipeline["PipelineArn"])
print("Pipeline registration saved. No execution started.")

Created: aai540-group3-ticket-routing-pipeline
Status: Active
PASS: SageMaker's saved definition matches the local definition.
Pipeline ARN: arn:aws:sagemaker:us-east-1:390568313988:pipeline/aai540-group3-ticket-routing-pipeline
Pipeline registration saved. No execution started.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import boto3

PROJECT_ROOT = (
    Path.home()
    / "AAI-540-Project"
    / "multilingual-it-ticket-routing-mlops"
)
assert PROJECT_ROOT.is_dir(), f"Project folder not found: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)

config_path = PROJECT_ROOT / "project_config.json"
project_config = json.loads(config_path.read_text())

AWS_REGION = project_config["aws_region"]
PROJECT_BUCKET = project_config["s3_bucket"]

aws_session = boto3.Session(region_name=AWS_REGION)
s3_client = aws_session.client("s3")
sagemaker_client = aws_session.client("sagemaker")

PIPELINE_SOURCE_DIR = PROJECT_ROOT / "src" / "ticket_pipeline"
PIPELINE_REPORT_DIR = PROJECT_ROOT / "reports" / "pipeline"

pipeline_config = json.loads(
    (PIPELINE_SOURCE_DIR / "pipeline_config.json").read_text()
)
pipeline_staging = json.loads(
    (PIPELINE_REPORT_DIR / "staged_sources.json").read_text()
)

definition_path = PIPELINE_SOURCE_DIR / "pipeline_definition.json"
definition_text = definition_path.read_text()
pipeline_definition = json.loads(definition_text)
definition_hash = hashlib.sha256(
    definition_text.encode("utf-8")
).hexdigest()

assert (
    definition_hash == project_config["ml_pipeline"]["definition_sha256"]
), "Local pipeline definition differs from the saved checkpoint."

# Verify that the local scripts and configuration match the staged files.
for filename, reference in pipeline_staging["source_artifacts"].items():
    actual_hash = hashlib.sha256(
        (PIPELINE_SOURCE_DIR / filename).read_bytes()
    ).hexdigest()
    assert actual_hash == reference["sha256"], f"File changed: {filename}"

pipeline_name = pipeline_staging["pipeline_name"]
pipeline_role = pipeline_config["execution_role"]
pipeline_image = pipeline_config["training_image"]

saved_pipeline = sagemaker_client.describe_pipeline(
    PipelineName=pipeline_name
)

assert saved_pipeline["PipelineStatus"] == "Active"
assert saved_pipeline["RoleArn"] == pipeline_role
assert (
    json.loads(saved_pipeline["PipelineDefinition"]) == pipeline_definition
), "The AWS pipeline definition differs from the local definition."

# Check for an existing execution before starting anything.
execution_history = sagemaker_client.list_pipeline_executions(
    PipelineName=pipeline_name,
    MaxResults=10,
    SortOrder="Descending",
)["PipelineExecutionSummaries"]

print("Project folder:", PROJECT_ROOT)
print("Region:", AWS_REGION)
print("Pipeline:", pipeline_name)
print("Pipeline status:", saved_pipeline["PipelineStatus"])
print("PASS: local files and AWS pipeline definition match the checkpoint.")

if execution_history:
    print("Recent executions:")
    for execution in execution_history:
        print(
            execution["PipelineExecutionArn"],
            execution["PipelineExecutionStatus"],
        )
else:
    print("No previous pipeline executions found.")

print("Session restored. No jobs started.")

Project folder: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops
Region: us-east-1
Pipeline: aai540-group3-ticket-routing-pipeline
Pipeline status: Active
PASS: local files and AWS pipeline definition match the checkpoint.
No previous pipeline executions found.
Session restored. No jobs started.


In [2]:
import json
import uuid

# Reload the checkpoint so rerunning this cell reuses a saved execution.
project_config = json.loads(config_path.read_text())
pipeline_checkpoint = project_config["ml_pipeline"]

request_path = PIPELINE_REPORT_DIR / "first_execution_request.json"

execution_request = {
    "PipelineName": pipeline_name,
    "PipelineExecutionDisplayName": f"first-run-{definition_hash[:12]}",
    "PipelineExecutionDescription": (
        "First verified run: data checks, frozen model training, "
        "validation quality gate, and registration for manual review."
    ),
    "PipelineParameters": [
        {
            "Name": "MinimumValidationMacroF1",
            "Value": str(
                pipeline_config["quality_gate"]["minimum_validation_macro_f1"]
            ),
        }
    ],
    "ParallelismConfiguration": {
        "MaxParallelExecutionSteps": 1
    },
    "ClientRequestToken": str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            f"{pipeline_name}:{definition_hash}:first-execution",
        )
    ),
}

# Save the exact request BEFORE contacting AWS.
if request_path.exists():
    saved_request = json.loads(request_path.read_text())
    assert saved_request == execution_request, (
        "An earlier execution request has different settings. "
        "Share this message before continuing."
    )
else:
    request_path.write_text(
        json.dumps(execution_request, indent=2) + "\n"
    )

previous_execution = pipeline_checkpoint.get("first_execution", {})
execution_arn = previous_execution.get("execution_arn")

if execution_arn:
    print("Reusing the saved execution.")
else:
    response = sagemaker_client.start_pipeline_execution(
        **execution_request
    )
    execution_arn = response["PipelineExecutionArn"]

    # Save immediately so a notebook restart can recover this execution.
    pipeline_checkpoint["first_execution"] = {
        "execution_arn": execution_arn,
        "definition_sha256": definition_hash,
        "request_file": str(request_path.relative_to(PROJECT_ROOT)),
    }
    config_path.write_text(json.dumps(project_config, indent=2) + "\n")

execution_details = sagemaker_client.describe_pipeline_execution(
    PipelineExecutionArn=execution_arn
)

pipeline_checkpoint["first_execution"]["status"] = (
    execution_details["PipelineExecutionStatus"]
)
config_path.write_text(json.dumps(project_config, indent=2) + "\n")

print("Pipeline:", pipeline_name)
print("Execution ARN:", execution_arn)
print("Status:", execution_details["PipelineExecutionStatus"])
print("Execution checkpoint saved.")

Pipeline: aai540-group3-ticket-routing-pipeline
Execution ARN: arn:aws:sagemaker:us-east-1:390568313988:pipeline/aai540-group3-ticket-routing-pipeline/execution/un5qr81r60u7
Status: Executing
Execution checkpoint saved.


In [7]:
import json
from datetime import datetime, timezone

project_config = json.loads(config_path.read_text())
execution_arn = project_config["ml_pipeline"]["first_execution"]["execution_arn"]

execution_details = sagemaker_client.describe_pipeline_execution(
    PipelineExecutionArn=execution_arn
)

steps = []
request = {
    "PipelineExecutionArn": execution_arn,
    "SortOrder": "Ascending",
    "MaxResults": 100,
}

while True:
    page = sagemaker_client.list_pipeline_execution_steps(**request)
    steps.extend(page["PipelineExecutionSteps"])

    if not page.get("NextToken"):
        break
    request["NextToken"] = page["NextToken"]

status = execution_details["PipelineExecutionStatus"]
print("Pipeline status:", status)

for step in steps:
    print(f"{step['StepName']}: {step['StepStatus']}")

    if step.get("FailureReason"):
        print("  Failure reason:", step["FailureReason"])

    condition = step.get("Metadata", {}).get("Condition")
    if condition:
        print("  Quality gate outcome:", condition["Outcome"])

if execution_details.get("FailureReason"):
    print("Pipeline failure reason:", execution_details["FailureReason"])

snapshot = {
    "checked_at": datetime.now(timezone.utc).isoformat(),
    "execution_arn": execution_arn,
    "status": status,
    "failure_reason": execution_details.get("FailureReason"),
    "steps": steps,
}

(PIPELINE_REPORT_DIR / "first_execution_status.json").write_text(
    json.dumps(snapshot, indent=2, default=str) + "\n"
)

project_config["ml_pipeline"]["first_execution"]["status"] = status
config_path.write_text(json.dumps(project_config, indent=2) + "\n")

if status == "Executing":
    print("\nStill running. Rerun this status cell after few minutes.")
elif status == "Succeeded":
    print("\nPipeline finished. Ready to verify its reports and registered model.")
else:
    print("\nShare this output before starting another execution.")

Pipeline status: Succeeded
VerifyData: Succeeded
TrainModel: Succeeded
PrepareEvaluation: Succeeded
CheckValidationQuality: Succeeded
  Quality gate outcome: True
RegisterModelForReview: Succeeded

Pipeline finished. Ready to verify its reports and registered model.


In [8]:
import hashlib
import json
from urllib.parse import urlparse

project_config = json.loads(config_path.read_text())
run_checkpoint = project_config["ml_pipeline"]["first_execution"]
execution_arn = run_checkpoint["execution_arn"]

snapshot = json.loads(
    (PIPELINE_REPORT_DIR / "first_execution_status.json").read_text()
)
assert snapshot["execution_arn"] == execution_arn
assert snapshot["status"] == "Succeeded"

step_map = {step["StepName"]: step for step in snapshot["steps"]}

required_steps = [
    "VerifyData",
    "TrainModel",
    "PrepareEvaluation",
    "CheckValidationQuality",
    "RegisterModelForReview",
]
for name in required_steps:
    assert step_map[name]["StepStatus"] == "Succeeded", name

assert (
    step_map["CheckValidationQuality"]["Metadata"]["Condition"]["Outcome"]
    == "True"
)

run_report_dir = PIPELINE_REPORT_DIR / "first_execution"
run_report_dir.mkdir(parents=True, exist_ok=True)
report_artifacts = {}


def processing_output_uri(step_name, output_name):
    job_arn = step_map[step_name]["Metadata"]["ProcessingJob"]["Arn"]
    job = sagemaker_client.describe_processing_job(
        ProcessingJobName=job_arn.rsplit("/", 1)[-1]
    )
    assert job["ProcessingJobStatus"] == "Completed"

    outputs = {
        output["OutputName"]: output["S3Output"]["S3Uri"]
        for output in job["ProcessingOutputConfig"]["Outputs"]
    }
    return outputs[output_name].rstrip("/")


def save_s3_report(uri, filename):
    location = urlparse(uri)
    response = s3_client.get_object(
        Bucket=location.netloc,
        Key=location.path.lstrip("/"),
    )
    try:
        content = response["Body"].read()
    finally:
        response["Body"].close()

    assert response.get("VersionId") not in (None, "", "null")
    report = json.loads(content)

    (run_report_dir / filename).write_bytes(content)
    report_artifacts[filename] = {
        "uri": uri,
        "version_id": response["VersionId"],
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    return report


verification_uri = processing_output_uri("VerifyData", "verification")
evaluation_uri = processing_output_uri("PrepareEvaluation", "evaluation")

data_report = save_s3_report(
    verification_uri + "/data_verification.json",
    "data_verification.json",
)
evaluation_report = save_s3_report(
    evaluation_uri + "/evaluation.json",
    "evaluation.json",
)
model_assets = save_s3_report(
    evaluation_uri + "/model_assets.json",
    "model_assets.json",
)

# Confirm that the pipeline used the intended datasets.
assert data_report["status"] == "PASS"
assert data_report["ticket_ids_disjoint"] is True

for split in ("train", "validation"):
    for field in ("uri", "version_id", "sha256", "rows"):
        assert (
            data_report["datasets"][split][field]
            == pipeline_config["datasets"][split][field]
        ), f"{split}: {field} mismatch."

# Identify the training job and model package created by this execution.
training_job_name = (
    step_map["TrainModel"]["Metadata"]["TrainingJob"]["Arn"].rsplit("/", 1)[-1]
)
package_arn = (
    step_map["RegisterModelForReview"]["Metadata"]["RegisterModel"]["Arn"]
)

training_job = sagemaker_client.describe_training_job(
    TrainingJobName=training_job_name
)
package = sagemaker_client.describe_model_package(
    ModelPackageName=package_arn
)

assert training_job["TrainingJobStatus"] == "Completed"
assert package["ModelPackageStatus"] == "Completed"
assert package["ModelApprovalStatus"] == "PendingManualApproval"
assert (
    package["ModelPackageGroupName"]
    == pipeline_config["model_package_group_name"]
)

assert evaluation_report["training_job_name"] == training_job_name
assert model_assets["training_job_name"] == training_job_name
assert model_assets["training_configuration_verified"] is True
assert evaluation_report["evaluation_split"] == "validation"
assert (
    evaluation_report["evaluated_tickets"]
    == pipeline_config["datasets"]["validation"]["rows"]
)

containers = package["InferenceSpecification"]["Containers"]
assert len(containers) == 1
container = containers[0]

assert (
    container["ModelDataUrl"]
    == training_job["ModelArtifacts"]["S3ModelArtifacts"]
    == model_assets["model_artifact"]["uri"]
)
assert container["Image"] == pipeline_config["serving_assets"]["serving_image"]
assert (
    container["Environment"]
    == pipeline_config["serving_assets"]["serving_environment"]
)
assert (
    package["ModelMetrics"]["ModelQuality"]["Statistics"]["S3Uri"]
    == report_artifacts["evaluation.json"]["uri"]
)

# Independently confirm the two gate conditions from the saved report.
metrics = {
    name: evaluation_report["multiclass_classification_metrics"][name]["value"]
    for name in ("accuracy", "macro_f1", "weighted_f1")
}
assert (
    metrics["macro_f1"]
    >= pipeline_config["quality_gate"]["minimum_validation_macro_f1"]
)
assert metrics["accuracy"] > evaluation_report["benchmark"]["accuracy"]

summary = {
    "execution_arn": execution_arn,
    "status": "Succeeded",
    "training_job_name": training_job_name,
    "model_package_arn": package_arn,
    "model_package_version": package["ModelPackageVersion"],
    "approval_status": package["ModelApprovalStatus"],
    "validation_metrics": metrics,
    "model_artifact": model_assets["model_artifact"],
    "report_artifacts": report_artifacts,
    "verification_passed": True,
}

(run_report_dir / "verification_summary.json").write_text(
    json.dumps(summary, indent=2) + "\n"
)

run_checkpoint.update(summary)
config_path.write_text(json.dumps(project_config, indent=2) + "\n")

print("PASS: pipeline datasets match the saved training and validation files.")
print("PASS: registered model and validation report belong to this execution.")
print("PASS: serving image and code configuration match.")
print("PASS: both validation quality conditions are satisfied.")

for name, value in metrics.items():
    print(f"Validation {name}: {value:.4f}")

print("Registered model version:", package["ModelPackageVersion"])
print("Approval status:", package["ModelApprovalStatus"])
print("Reports saved:", run_report_dir)
print("No new jobs started.")

PASS: pipeline datasets match the saved training and validation files.
PASS: registered model and validation report belong to this execution.
PASS: serving image and code configuration match.
PASS: both validation quality conditions are satisfied.
Validation accuracy: 0.4557
Validation macro_f1: 0.4362
Validation weighted_f1: 0.4569
Registered model version: 2
Approval status: PendingManualApproval
Reports saved: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/reports/pipeline/first_execution
No new jobs started.
